## Scan companies with new configs

- Use Search tools to find companies
- Run LLM verification
- Produce and upload the new tables and viz landscapes

In [ ]:
from discovery_utils.getters import gtr
from discovery_utils.getters import crunchbase
from discovery_utils.utils import search

from discovery_utils.utils.llm.batch_check import LLMProcessor, generate_relevance_check_system_message

from src import PROJECT_DIR
from src import VECTOR_DB_DIR

OUTPUT_DIR = PROJECT_DIR / 'data/2025_02_MS_ai_healthcare/'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCORE_THRESHOLD = 0.3

configs = ["ai_healthcare"]
config_files = {config: f"config_{config}.yaml" for config in configs}

In [ ]:
CB = crunchbase.CrunchbaseGetter(vector_db_path=VECTOR_DB_DIR)

## Find companies

In [ ]:
for config in configs:
    config_file = config_files[config]
    # keyword + vector search
    SearchCB = search.SearchDataset(CB, CB.organisations_enriched, config_file)
    search_cb_df = (
        SearchCB.do_search()
        # add full description text
        .merge(CB.descriptions[['id', 'description']], on='id', how='left')
        .fillna({'description': '', 'short_description': '', 'name': ''})
        .assign(text = lambda df: df['name'] + '. ' + df['short_description'] + ' ' + df['description'])
    )    
    # Filter the results to only include those with a score above a threshold
    relevant_df = search_cb_df.query(f"_score_avg > {SCORE_THRESHOLD}")
    relevant_df.to_csv(OUTPUT_DIR / f"relevant_{config}.csv", index=False)

    system_message = generate_relevance_check_system_message(config_file)
    fields = [
        {"name": "is_relevant", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},
    ]
    check_data = dict(zip(relevant_df['id'], relevant_df['text']))

    processor = LLMProcessor(
        output_path=str(OUTPUT_DIR / f"llm_check_MS_{config}.jsonl"),
        system_message=system_message,
        session_name="mission_studio",
        output_fields=fields,
    )

    await processor.run(check_data, batch_size=15, sleep_time=0.5)   

## Produce stats and upload on Google 
- Load LLM check results
- Produce stats
- Repeat the search
- Merge data
- Create landscapes
- Upload on Google Slides and Google Sheets


In [ ]:
from discovery_utils.utils import (
    analysis_crunchbase,
    analysis,
    charts,
    google,
    google_slides,
    viz_landscape,
)
import pandas as pd
from src import logging

FUNDING_ROUND_TYPES = ["angel", "pre_seed", "seed", "series_a", "series_b"]
FUNDING_ROUND_TYPES_STR = "Angel, Pre-seed, Seed, Series A, Series B"

In [ ]:
cols_funding_rounds = [
    "funding_round_name", 
    "org_name",
    "theme",
    "cb_url",
    "country_code", 
    "region_nesta",
    "region",
    "city",
    "year",
    "announced_on",
    "investment_type", 
    "investment_stage",
    "raised_amount_gbp",
    "raised_amount_usd",
    "raised_amount", 
    "raised_amount_currency_code",
    "post_money_valuation_usd", 
    "post_money_valuation",
    "post_money_valuation_currency_code", 
    "investor_name",
]

In [ ]:
cols_companies = [
    "name", 
    "short_description", 
    "founded_on", 
    'created_at',    
    "cb_url", 
    "homepage_url", 
    "ai_relevance_check",
    "theme",
    'landscape_category',
    'landscape_keyword_cluster',
    'mission_labels',
    'topic_labels',
    "rank", 
    "country_code", 
    'region_nesta',
    "region", 
    "city", 
    "status", 
    "category_list", 
    "closed_on", 
    "employee_count", 
    "email", 
    "phone", 
    "facebook_url", 
    "linkedin_url", 
    "twitter_url", 
    "logo_url", 
    'recent_funding',    
    "num_exits", 
    "num_funding_rounds", 
    "last_funding_on", 
    "investment_funding_gbp", 
    "num_investment_rounds", 
    "grant_funding_gbp", 
    "num_grants", 
    "total_funding_gbp", 
    "smart_money", 
    "_score_keywords",
    "_score_vectors",
    "_score_avg",
]

In [ ]:
def produce_stats(CB, matching_ids: list[str], category_name: str) -> None:
    """ Produce stats for the companies and output charts """ 
    # Check companies by querying ids
    matchings_orgs_df = CB.organisations_enriched.query("id in @matching_ids")

    # Get the funding rounds for the matching companies
    funding_rounds_df = (
        CB.select_funding_rounds(org_ids=matching_ids, funding_round_types=FUNDING_ROUND_TYPES)
    )

    # organise investors by each funding round
    investors_df = (
        CB.funding_rounds_enriched
        .query("funding_round_id in @funding_rounds_df.funding_round_id")
        .groupby("funding_round_id")
        .agg(investor_name=("investor_name", list))
        .reset_index()
    )

    funding_rounds_df = (
        funding_rounds_df
        .drop(columns=["investor_name"])
        .merge(investors_df, on="funding_round_id", how="left")
    )

    # generate basic time series
    ts_df = analysis_crunchbase.get_timeseries(
        matchings_orgs_df, 
        funding_rounds_df, 
        period='year', 
        min_year=2014, 
        max_year=2025
    )
    growth_rates = analysis.smoothed_growth(ts_df, year_start=2020, year_end=2024)
    growth_rates_df = pd.DataFrame(growth_rates, columns=[category_name]).T.reset_index().rename(columns={'index': 'theme'})  

    # Let's look into breakdown of deal types
    # deals_df, deal_counts_df = analysis_crunchbase.get_funding_by_year_and_range(funding_rounds_df, 2014, 2025)
    aggregated_funding_types_df = analysis_crunchbase.aggregate_by_funding_round_types(funding_rounds_df)

    # IPOs and acquisitions
    ipos_df = CB.ipos.query("org_id in @matching_ids")
    acquisitions_df = CB.acquisitions.query("acquiree_id in @matching_ids")

    if len(ipos_df) > 0:
        ipos_df.to_csv(OUTPUT_DIR / f"ipos_{category_name}.csv", index=False)

    if len(acquisitions_df) > 0:
        acquisitions_df.to_csv(OUTPUT_DIR / f"acquisitions_{category_name}.csv", index=False)
        

    # Fig variables
    prefix = f"{OUTPUT_DIR}/charts/{category_name}_"
    _scale = 2

    # Investment amounts (total)
    fig = charts.ts_bar(
        ts_df,
        variable='raised_amount_gbp_total',
        variable_title="Raised amount, £ millions",
        category_column="_category",
    )
    fig = charts.configure_plots(fig, chart_title=f"Funding raised over time for {category_name}")
    chart_filename = f"{prefix}raised_amount.png"
    fig.save(chart_filename, scale_factor=_scale)

    # Number of companies
    fig = charts.ts_bar(
        ts_df,
        variable='n_orgs_founded',
        variable_title="Number of companies founded",
        category_column="_category",
    )
    fig = charts.configure_plots(fig, chart_title=f"Number of founded {category_name} companies")
    chart_filename = f"{prefix}no_of_companies.png"
    fig.save(chart_filename, scale_factor=_scale)    

    # Investment amounts (by type)
    investment_types_fig = analysis_crunchbase.chart_investment_types(aggregated_funding_types_df)
    investment_types_fig = charts.configure_plots(investment_types_fig, chart_title=f"Breakdown of investment types for {category_name}")
    investment_types_chart_filename = f"{prefix}investment_types.png"
    investment_types_fig.save(investment_types_chart_filename, scale_factor=_scale)

    # Investment counts (by type)
    investment_types_counts_fig = analysis_crunchbase.chart_investment_types_counts(aggregated_funding_types_df)
    investment_types_counts_fig = charts.configure_plots(investment_types_counts_fig, chart_title=f"Number of investments by type for {category_name}")
    investment_types_counts_chart_filename = f"{prefix}investment_types_counts.png"
    investment_types_counts_fig.save(investment_types_counts_chart_filename, scale_factor=_scale)

    # deal_sizes_fig = analysis_crunchbase.chart_deal_sizes(deals_df)
    # deal_sizes_fig = charts.configure_plots(deal_sizes_fig, chart_title=f"Deal sizes for {category_name}")
    # deal_sizes_chart_filename = f"{prefix}deal_sizes.png"
    # deal_sizes_fig.save(deal_sizes_chart_filename, scale_factor=_scale)

    # deal_sizes_counts_fig = analysis_crunchbase.chart_deal_sizes_counts(deal_counts_df)
    # deal_sizes_counts_fig = charts.configure_plots(deal_sizes_counts_fig, chart_title=f"Number of deals by size for {category_name}")
    # deal_sizes_counts_chart_filename = f"{prefix}deal_sizes_counts.png"
    # deal_sizes_counts_fig.save(deal_sizes_counts_chart_filename, scale_factor=_scale)

    return ts_df, growth_rates_df, ipos_df, acquisitions_df, matchings_orgs_df, funding_rounds_df


In [ ]:
all_ts_df = []
all_ipos_df = []
all_acquisitions_df = []
all_growth_rates = []
all_export_df = []
all_funding_rounds_df = []
all_orgs = []
all_viz_df = []

for config_name in configs:
    logging.info(f"Processing {config_name}")
    # load in the LLM results
    relevant_df = pd.read_csv(OUTPUT_DIR / f"relevant_{config_name}.csv")
    relevant_check_df = pd.read_json(OUTPUT_DIR / f"llm_check_MS_{config_name}.jsonl", lines=True)
    relevant_checked_df = relevant_df.merge(relevant_check_df[['id', 'is_relevant']], left_on='id', right_on='id', how='left')
    matching_ids = relevant_checked_df.query("is_relevant == 'yes'").id.tolist()
    # matching_ids = relevant_checked_df.id.tolist()
    
    ts_df, growth_rates_df, ipos_df, acquisitions_df, matchings_orgs_df, funding_rounds_df = produce_stats(CB, matching_ids, config_name)

    all_ts_df.append(ts_df.assign(theme=config_name))
    all_growth_rates.append(growth_rates_df)
    all_ipos_df.append(ipos_df.assign(theme=config_name))
    all_acquisitions_df.append(acquisitions_df.assign(theme=config_name))  
    all_funding_rounds_df.append(funding_rounds_df.assign(theme=config_name))
    all_orgs.append(matchings_orgs_df.assign(theme=config_name))    

    # Landscapes
    
    id_condition = "id in ('{}')".format("', '".join(list(matching_ids)))
    vectors_df = CB.VectorDB.vector_db.search().where(id_condition).limit(30000).to_pandas()    

    fig, cb_viz_df = viz_landscape.generate_crunchbase_landscape(vectors_df, CB, min_cluster_size=15, n_keyword_clusters=10)

    output_path = f"cb_landscape_{config_name}.html"
    fig.save(str(output_path))    
    export_df = (
        relevant_checked_df
        .merge(cb_viz_df[['id', 'category', 'keyword_cluster', 'recent_funding', 'region']].rename(columns={"region": "nesta_region"}), on='id', how='left')
    )   
    export_df.to_csv(OUTPUT_DIR / f"cb_landscape_data_{config_name}.csv", index=False)
    
    all_viz_df.append(export_df.assign(theme=config_name))


In [ ]:
relevant_check_df

In [ ]:
all_ts_df = pd.concat(all_ts_df, ignore_index=True)
all_growth_rates = pd.concat(all_growth_rates, ignore_index=True)
all_ipos_df = pd.concat(all_ipos_df, ignore_index=True)
all_acquisitions_df = pd.concat(all_acquisitions_df, ignore_index=True)
all_funding_rounds_df = pd.concat(all_funding_rounds_df, ignore_index=True)
all_orgs = pd.concat(all_orgs, ignore_index=True)
all_viz_df = pd.concat(all_viz_df, ignore_index=True)


In [ ]:
all_ts_df.to_csv(OUTPUT_DIR / "all_ts_df.csv", index=False)
all_growth_rates.to_csv(OUTPUT_DIR / "all_growth_rates.csv", index=False)
all_ipos_df.to_csv(OUTPUT_DIR / "all_ipos_df.csv", index=False)
all_acquisitions_df.to_csv(OUTPUT_DIR / "all_acquisitions_df.csv", index=False)
all_funding_rounds_df.to_csv(OUTPUT_DIR / "all_funding_rounds_df.csv", index=False)
all_orgs.to_csv(OUTPUT_DIR / "all_orgs.csv", index=False)
all_viz_df.to_csv(OUTPUT_DIR / "all_viz_df.csv", index=False)

## Google uploads

In [ ]:
all_ts_df = pd.read_csv(OUTPUT_DIR / "all_ts_df.csv")
all_growth_rates = pd.read_csv(OUTPUT_DIR / "all_growth_rates.csv")
all_ipos_df = pd.read_csv(OUTPUT_DIR / "all_ipos_df.csv")
all_acquisitions_df = pd.read_csv(OUTPUT_DIR / "all_acquisitions_df.csv")
all_funding_rounds_df = pd.read_csv(OUTPUT_DIR / "all_funding_rounds_df.csv")
all_orgs = pd.read_csv(OUTPUT_DIR / "all_orgs.csv")
all_viz_df = pd.read_csv(OUTPUT_DIR / "all_viz_df.csv")


In [ ]:
_all_funding_rounds_df = (
    all_funding_rounds_df
    .assign(investment_stage = lambda df: df.investment_type.map(crunchbase.investment_type_to_stage()))
    .assign(region_nesta = lambda df: df.country_code.map(crunchbase.country_to_region()))
)[cols_funding_rounds]

In [ ]:
_all_orgs = (
    all_viz_df
    .rename(columns={'category': 'landscape_category', 'keyword_cluster': 'landscape_keyword_cluster', "is_relevant": "ai_relevance_check"})
    .assign(region_nesta = lambda df: df.country_code.map(crunchbase.country_to_region()))
)[cols_companies]

In [ ]:
len(vectors_df)

In [ ]:
sheet_id = "1PeI0hPsTPNnd0wYm9irEgIxI91moHaitWLxTPlq2BsE"

google.upload_data_to_gsheet(sheet_id, {"crunchbase_companies": _all_orgs})
google.format_gsheet(sheet_id, "crunchbase_companies", freeze_cols=4)

google.upload_data_to_gsheet(sheet_id, {"crunchbase_funding": _all_funding_rounds_df})
google.format_gsheet(sheet_id, "crunchbase_funding", freeze_cols=2)

google.upload_data_to_gsheet(sheet_id, {"crunchbase_aquisitions": all_acquisitions_df})
google.format_gsheet(sheet_id, "crunchbase_aquisitions", freeze_cols=0)

google.upload_data_to_gsheet(sheet_id, {"crunchbase_ipos": all_ipos_df})
google.format_gsheet(sheet_id, "crunchbase_ipos", freeze_cols=0)


## Google slides

In [ ]:
gdrive_service = google_slides.get_drive_service()
gslides_service = google_slides.get_slides_service()

In [ ]:
PRESENTATION_ID = "18EmhSxtPT0OqG_woX2ZTP909rrbD6Fyv6QOqANIJ_gI"
TEMPLATE_SLIDE = "g33a945c2053_0_666"

In [ ]:
# for config_name, category in src_utils.CB_CATEGORIES.items():
all_file_ids = []

for category in configs:
    logging.info(f"Processing {category}")
    all_requests = []

    slide_id = category.lower().replace(" ", "_")

    # Investment amounts
    fig_path = f"{OUTPUT_DIR}/charts/{category}_investment_types.png"
    file_id, image_url = google_slides.upload_image_to_drive(gdrive_service, fig_path)
    all_file_ids.append(file_id)

    text_funding = {
        "chart_type": "funding",
        "investment_types": FUNDING_ROUND_TYPES_STR,
        "category": f"{category}",
        "n_companies": len(all_orgs.query("theme == @category").drop_duplicates("id")),
        "n_rounds": len(all_funding_rounds_df.query("theme == @category").drop_duplicates("funding_round_id")),
        "year_start": 2014,
        "year_end": 2025,
        "growth": round(all_growth_rates.query("theme == @category")['raised_amount_gbp_total'].iloc[0],0),
        "growth_start": 2020,
        "growth_end": 2024,
        "baseline_growth": 41,
    }

    all_requests += (
        google_slides
        .MissionStudioTemplate(
            template_id = TEMPLATE_SLIDE,
            slide_id = slide_id + "_funding",
            image_url = image_url,
            heading_text = category,
            details_text = google_slides.text_investment(**text_funding)
        )
        .slide_request()
    )

    # Number of investment rounds
    fig_path = f"{OUTPUT_DIR}/charts/{category}_investment_types_counts.png"
    file_id, image_url = google_slides.upload_image_to_drive(gdrive_service, fig_path)
    all_file_ids.append(file_id)

    text_rounds = {
        "chart_type": "number_of_rounds",
        "investment_types": FUNDING_ROUND_TYPES_STR,
        "category": f"{category}",
        "n_companies": len(all_orgs.query("theme == @category").drop_duplicates("id")),
        "n_rounds": len(all_funding_rounds_df.query("theme == @category").drop_duplicates("funding_round_id")),
        "year_start": 2014,
        "year_end": 2025,
        "growth": round(all_growth_rates.query("theme == @category")['n_rounds'].iloc[0],0),
        "growth_start": 2020,
        "growth_end": 2024,
        "baseline_growth": -1.2,
    }

    all_requests += (
        google_slides
        .MissionStudioTemplate(
            template_id = TEMPLATE_SLIDE,
            slide_id = slide_id + "_rounds",
            image_url = image_url,
            heading_text = category,
            details_text = google_slides.text_investment(**text_rounds)
        )
        .slide_request()
    )

    # Number of companies
    slide_id = category.lower().replace(" ", "_")
    fig_path = f"{OUTPUT_DIR}/charts/{category}_no_of_companies.png"
    file_id, image_url = google_slides.upload_image_to_drive(gdrive_service, fig_path)
    all_file_ids.append(file_id)

    text_new_companies = {
        "chart_type": "number_of_new_companies",
        "investment_types": FUNDING_ROUND_TYPES_STR,
        "category": f"{category}",
        "n_companies": len(all_orgs.query("theme == @category").drop_duplicates("id")),
        "n_rounds": len(all_funding_rounds_df.query("theme == @category").drop_duplicates("funding_round_id")),
        "year_start": 2014,
        "year_end": 2025,
        "growth": round(all_growth_rates.query("theme == @category")['n_orgs_founded'].iloc[0],0),
        "growth_start": 2020,
        "growth_end": 2024,
        "baseline_growth": -71,
    }


    all_requests += (
        google_slides
        .MissionStudioTemplate(
            template_id = TEMPLATE_SLIDE,
            slide_id = slide_id + "_companies",
            image_url = image_url,
            heading_text = category,
            details_text = google_slides.text_investment(**text_new_companies)
        )
        .slide_request()
    )

    response = (
        gslides_service
        .presentations()
        .batchUpdate(presentationId=PRESENTATION_ID, body={"requests": all_requests})
        .execute()
    )  

In [ ]:
for file_id in all_file_ids:
    try:
        google.delete_file_from_drive(gdrive_service, file_id)      
    except Exception as e:
        logging.error(f"Error deleting file: {e}")

## Gateway to Research

In [ ]:
from discovery_utils.utils import analysis_gtr

In [ ]:
GTR = gtr.GtrGetter(vector_db_path=VECTOR_DB_DIR, data_version = "GtR_20250223")

In [ ]:
for config in configs:
    config_file = config_files[config]
    # keyword + vector search
    SearchGTR= search.SearchDataset(GTR, GTR.projects_enriched, config_file)
    search_df = (
        SearchGTR.do_search()
        # add full description text
        .merge(GTR.get_projects_text()[['id', 'text']])
    )    
    # Filter the results to only include those with a score above a threshold
    relevant_df = search_df.query(f"_score_avg > {SCORE_THRESHOLD}")
    relevant_df.to_csv(OUTPUT_DIR / f"relevant_gtr_{config}.csv", index=False)

    system_message = generate_relevance_check_system_message(config_file)
    fields = [
        {"name": "is_relevant", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},
    ]
    check_data = dict(zip(relevant_df['id'], relevant_df['text']))

    processor = LLMProcessor(
        output_path=str(OUTPUT_DIR / f"llm_check_MS_gtr_{config}.jsonl"),
        system_message=system_message,
        session_name="mission_studio",
        output_fields=fields,
    )

    await processor.run(check_data, batch_size=15, sleep_time=0.5)   

In [ ]:
relevant_df = pd.read_csv(OUTPUT_DIR / f"relevant_gtr_{config_name}.csv")
relevant_check_df = pd.read_json(OUTPUT_DIR / f"llm_check_MS_gtr_{config_name}.jsonl", lines=True)
relevant_checked_df = relevant_df.merge(relevant_check_df[['id', 'is_relevant']], left_on='id', right_on='id', how='left')
matching_ids = relevant_checked_df.query("is_relevant == 'yes'").id.tolist()

In [ ]:
len(matching_ids)

In [ ]:
cols_projects = [
    'title',
    "ai_relevance_check",      
    'status', 
    'grantCategory',
    'leadFunder',
    'abstractText',
    'techAbstractText',
    'potentialImpact',
    'start',
    'end',
    'amount',
    'url',
    "landscape_category",
    "landscape_keyword_cluster",
    "_score_keywords",
    "_score_vectors",
    "_score_avg",    
]

In [ ]:
len(relevant_checked_df)

In [ ]:
ts_df = (
    analysis_gtr.get_timeseries(relevant_checked_df, period='year', min_year=2010, max_year=2025, description_column="abstractText")
    .assign(amount = lambda df: df.amount / 1_000_000)
)
fig = charts.ts_bar(
    ts_df,
    variable='n_projects',
    variable_title="Number of projects",
    category_column="_category",
)
charts.configure_plots(fig, chart_title="")

In [ ]:
fig = charts.ts_bar(
    ts_df,
    variable='amount',
    variable_title="Amount, £ millions",
    category_column="_category",
)
charts.configure_plots(fig, chart_title="")

In [ ]:
# write an sql query to achieve id in test_ids
id_condition = "id in ('{}')".format("', '".join(list(matching_ids)))
vectors_df = GTR.VectorDB.vector_db.search().where(id_condition).limit(30000).to_pandas()
len(vectors_df)

In [ ]:
fig, gtr_viz_df = viz_landscape.generate_gtr_landscape(vectors_df, GTR, min_cluster_size=25,  verbose=True)

In [ ]:
save_name = config_name
output_path = OUTPUT_DIR / f'landscape_gtr_{save_name}.html'
# gtr_viz_df.to_csv(PROJECT_DIR / f"data/2025_01_MS_ahl/_table_gtr_{save_name}.csv", index=False)
fig.save(str(output_path))

In [ ]:
_df = (
    relevant_checked_df
    .merge(
        gtr_viz_df[['id', 'category', 'keyword_cluster']].rename(columns={"category": "landscape_category", "keyword_cluster": "landscape_keyword_cluster", "region": "region_nesta"}), on='id', how='left')
    .rename(columns = {"is_relevant": "ai_relevance_check"})
)[cols_projects]
_df

In [ ]:
google.upload_data_to_gsheet(sheet_id, {"ukri_projects": _df})
google.format_gsheet(sheet_id, "ukri_projects", freeze_cols=1)